# 冰水主機台數控制

說明冰水主機台數控制規則及其依據：以兩台性能曲線比較各運轉模式的效率，並以實測資料確認單機容量上限。

台數依預測冷負荷與單機容量決定，不以效率最佳化決定（規則見第 4 節）。第 3 節的「最佳分配」為理論對照值。

| 主機 | 機型 | 額定 | 性能曲線（kW/RT 對負載率） | 最低負載 |
|---|---|---|---|---|
| CH-1 | Trane 螺旋定頻（2008） | 375 RT | a + b ÷ 負載率 | 20%（75 RT） |
| CH-2 | 堃霖磁浮變頻（2024） | 400 RT | 三次多項式 | 15%（60 RT） |

- 曲線用穩定運轉、負載率 ≤ 100% 的資料擬合
- 超出實測負載率範圍標為「外推」

## 1. 環境設定

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# 切到專案根目錄，才 import 得到 data、model、research
if Path.cwd().name == "analysis":
    os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import settings
from data.sensor import is_running  # 運轉：功率 > 20 kW 且冰水流量 > 100 LPM
from data.sensor import load_chiller
from model import rules
from model.train import get_chiller_hourly
from research import chiller
from research.ch2_model import WARMUP_MIN
from research.chart_style import show

pio.renderers.default = "jupyterlab"

# 資料期間
START_TS = int(pd.Timestamp("2026-09-17", tz="Asia/Taipei").timestamp())
END_TS = int(pd.Timestamp("2026-10-14", tz="Asia/Taipei").timestamp())

## 2. 性能曲線

In [2]:
# 兩台每分鐘原始資料（後面各節共用），並擬合性能曲線
raw_minutes = {}
for name, chiller_config in rules.CHILLERS.items():
    raw_minutes[name] = await load_chiller(chiller_config["id"])
curves = {}
for name in rules.CHILLERS:
    curves[name] = chiller.fit_curve(raw_minutes[name], name)


# 係數文字；極小值寫成 ×10ⁿ
def coefficient_text(value: float) -> str:
    if value != 0 and abs(value) < 0.001:
        mantissa, exponent = f"{value:.2e}".split("e")
        return f"{mantissa}×10<sup>{int(exponent)}</sup>"
    return f"{value:.3g}"


# kW/RT 擬合式文字（x：負載率 %）
def poly_text(curve: dict) -> str:
    if curve["x"] == "1/LR":
        b, a = curve["coef"]
        return f"{coefficient_text(a)} + {coefficient_text(b)}/x"
    degree = len(curve["coef"]) - 1
    terms = []
    for i, value in enumerate(curve["coef"]):
        power = degree - i
        if power == 0:
            variable = ""
        elif power == 1:
            variable = "x"
        else:
            variable = f"x<sup>{power}</sup>"
        if value < 0:
            sign = "−"
        else:
            sign = "+"
        terms.append(f"{sign} {coefficient_text(abs(value))}{variable}")
    text = " ".join(terms)
    # 第一項是正數時不寫「+」
    if text.startswith("+"):
        return text[2:]
    return "−" + text[2:]


curve_rows = []
for name, curve in curves.items():
    lr_grid = np.arange(curve["lr_min"], chiller.MAX_LR + 0.01, 0.5)
    curve_kw_per_rt = chiller.kw_per_rt(curve, lr_grid)
    equation_text = poly_text(curve).replace("<sup>", "^").replace("</sup>", "")
    curve_rows.append(
        {
            "主機": name,
            "額定容量 (RT)": curve["rated"],
            "擬合式（x：負載率 %）": "kW/RT = " + equation_text,
            "R²": round(curve["r2"], 3),
            "資料負載率範圍 (%)": f"{curve['lr_min']:.0f}~{curve['lr_max']:.0f}",
            "樣本數（分鐘）": curve["n"],
            "最佳效率負載率 (%)": round(lr_grid[curve_kw_per_rt.argmin()]),
            "最低 kW/RT": round(curve_kw_per_rt.min(), 3),
            "滿載 kW/RT": round(float(chiller.kw_per_rt(curve, 100)), 3),
        }
    )
pd.DataFrame(curve_rows).set_index("主機")

,額定容量 (RT),擬合式（x：負載率 %）,R²,資料負載率範圍 (%),樣本數（分鐘）,最佳效率負載率 (%),最低 kW/RT,滿載 kW/RT
主機,,,,,,,,
CH-1,375,kW/RT = 0.305 + 35.9/x,0.978,24~83,1693,100,0.665,0.664
CH-2,400,kW/RT = −6.33×10^-6x^3 + 0.00139x^2 − 0.0958x ...,0.692,46~100,7148,55,0.473,0.599


In [3]:
RT_TO_KW = 3.517  # COP = 3.517 ÷ kW/RT
# （實測點, 曲線）
SHADES = {
    "CH-1": ("#86b6ef", "#1c5cab"),
    "CH-2": ("#f5a582", "#a33f17"),
}
GROUP = {"CH-1": "legend", "CH-2": "legend2"}  # 圖例分組
# 公式位置（放在空白角落）
NOTE = {
    "CH-1": (99, 1.2, "right"),
    "CH-2": (21, 8.6, "left"),
}
HOVER = "負載率 %{x:.1f}%｜COP %{y:.2f}"
lr_grid = np.arange(20, chiller.MAX_LR + 0.1, 0.5)

fig = go.Figure()
for name, curve in curves.items():
    light, dark = SHADES[name]

    # 擬合曲線：資料範圍內實線、外推虛線
    cop = RT_TO_KW / chiller.kw_per_rt(curve, lr_grid)
    cop[cop <= 0] = np.nan
    inside = (lr_grid >= curve["lr_min"]) & (lr_grid <= curve["lr_max"])
    fig.add_trace(
        go.Scatter(
            x=lr_grid,
            y=np.where(inside, cop, np.nan),
            mode="lines",
            name="擬合曲線",
            legend=GROUP[name],
            line=dict(color=dark, width=2.5),
            hovertemplate=HOVER + "<extra>擬合</extra>",
        )
    )
    fig.add_trace(
        go.Scatter(
            x=lr_grid,
            y=np.where(inside, np.nan, cop),
            mode="lines",
            name="外推",
            legend=GROUP[name],
            showlegend=False,
            line=dict(color=dark, width=2, dash="dash"),
            hovertemplate=HOVER + "<extra>外推</extra>",
        )
    )

    # 實測：每分鐘點；每 5% 負載率區間平均（總冷凍噸 ÷ 總耗電）
    steady_data = chiller.steady(raw_minutes[name], curve["rated"])
    steady_data = steady_data[steady_data["LR"] <= chiller.MAX_LR]
    fig.add_trace(
        go.Scattergl(
            x=steady_data["LR"],
            y=RT_TO_KW / steady_data["KW_PER_RT"],
            mode="markers",
            name="實測",
            legend=GROUP[name],
            marker=dict(size=4, color=light, opacity=0.25),
            text=steady_data.index.strftime("%m/%d %H:%M"),
            hovertemplate="%{text}｜" + HOVER + "<extra></extra>",
        )
    )
    lr_bins = pd.cut(steady_data["LR"], np.arange(0, chiller.MAX_LR + 1, 5))
    lr_groups = steady_data.groupby(lr_bins, observed=True)
    bin_centers = [load_bin.left + 2.5 for load_bin in lr_groups.size().index]
    fig.add_trace(
        go.Scatter(
            x=bin_centers,
            y=RT_TO_KW * lr_groups["Q_COOL"].sum() / lr_groups["kW"].sum(),
            mode="markers",
            name="區間平均",
            legend=GROUP[name],
            showlegend=False,
            marker=dict(
                size=9,
                color=dark,
                symbol="diamond",
                line=dict(width=1, color="white"),
            ),
            text=lr_groups.size().astype(str),
            hovertemplate=HOVER + "｜實測 %{text} 筆<extra>5% 區間平均</extra>",
        )
    )

    # 公式（x：負載率 %，y：COP）；R²、CV(RMSE) 以 kW/RT 計算
    residual = steady_data["KW_PER_RT"] - chiller.kw_per_rt(curve, steady_data["LR"])
    cv_rmse = np.sqrt((residual**2).mean()) / steady_data["KW_PER_RT"].mean() * 100
    note_x, note_y, anchor = NOTE[name]
    fig.add_annotation(
        x=note_x,
        y=note_y,
        text=(
            f"<b>{name}</b>　y = 3.517 / ({poly_text(curve)})"
            f"<br>R² {curve['r2']:.2f}、CV(RMSE) {cv_rmse:.1f}%（kW/RT 擬合）"
        ),
        showarrow=False,
        xanchor=anchor,
        align=anchor,
        font=dict(size=13, color=dark),
        bgcolor="rgba(255,255,255,0.85)",
    )

fig.update_xaxes(title_text="負載率 (%)")
fig.update_yaxes(title_text="COP", range=[0, 9])
show(
    fig,
    "冰水主機性能曲線",
    "虛線為外推",
    legend_groups=["CH-1（375 RT 螺旋定頻）", "CH-2（400 RT 磁浮變頻）"],
    height=850,
)

## 3. 負荷與耗電

In [4]:
# 全館逐時負荷（CH-1 + CH-2）
load_hourly = await get_chiller_hourly(START_TS, END_TS)
load_hourly.index = pd.to_datetime(
    load_hourly["ts_hour"],
    unit="s",
    utc=True,
).dt.tz_convert(settings.settings.timezone)
actual_load = load_hourly["Q_COOL"]

SHUTDOWN_MIN = 10  # 停機前幾分鐘不算穩定運轉


# 穩定運轉的分鐘：開機滿 WARMUP_MIN、距停機 ≥ SHUTDOWN_MIN，且負荷 > 50 RT
def steady_minutes(minutes: pd.DataFrame) -> pd.Series:
    running = is_running(minutes)
    segment = (running != running.shift()).cumsum()  # 每次開／停機換一段
    run_min = running.groupby(segment).cumsum()  # 已運轉分鐘
    left_min = running.groupby(segment).transform("sum") - run_min  # 距停機分鐘
    return (
        running
        & (run_min > WARMUP_MIN)
        & (left_min >= SHUTDOWN_MIN)
        & (minutes["Q_COOL"] > 50)
    )


# 各機每分鐘資料（標記穩定運轉）
minute_data = {}
for name, raw in raw_minutes.items():
    minute_data[name] = raw.resample("1min").mean()
    minute_data[name]["STEADY"] = steady_minutes(minute_data[name])

# 實測點（每分鐘穩定運轉）：單機用該機負荷，兩台同時運轉用合計
minute_index = minute_data["CH-1"].index.union(minute_data["CH-2"].index)
ch1_minutes = minute_data["CH-1"].reindex(minute_index)
ch2_minutes = minute_data["CH-2"].reindex(minute_index)
ch1_steady = ch1_minutes["STEADY"].fillna(False).astype(bool)
ch2_steady = ch2_minutes["STEADY"].fillna(False).astype(bool)
both_steady = ch1_steady & ch2_steady
points = {  # 名稱：(負荷, 耗電, 顏色)
    "CH-2": (
        ch2_minutes.loc[ch2_steady, "Q_COOL"],
        ch2_minutes.loc[ch2_steady, "kW"],
        "#eb6834",
    ),
    "CH-1": (
        ch1_minutes.loc[ch1_steady, "Q_COOL"],
        ch1_minutes.loc[ch1_steady, "kW"],
        "#2a78d6",
    ),
    "CH-1＋CH-2": (
        ch1_minutes.loc[both_steady, "Q_COOL"] + ch2_minutes.loc[both_steady, "Q_COOL"],
        ch1_minutes.loc[both_steady, "kW"] + ch2_minutes.loc[both_steady, "kW"],
        "#8c564b",
    ),
}

# 各總負荷的建議運轉（最多兩台額定合計 775 RT；超過 PARALLEL_ABOVE 一律並聯）
candidate_loads = pd.Series(np.arange(50, 775.1, 1.0))
table = chiller.decide(
    candidate_loads,
    curves,
    parallel_above=rules.PARALLEL_ABOVE,
).set_index("全館負荷 RT")

### 3.1 單機容量上限：只開 CH-2 超過 380 RT 時出水拉不下來

只開 CH-2 時，出水拉不下來的小時都在負荷 382~396 RT，耗電 233~252 kW 已達型錄額定輸入（230.5 kW），主機沒有餘力把出水壓低，所以超過 380 RT 改兩台並聯。

In [5]:
# 只開 CH-2、穩定運轉的小時（穩定 ≥ 45 分鐘、CH-1 整小時停機）
ch2_minute_data = minute_data["CH-2"]
steady_count = ch2_minute_data["STEADY"].resample("1h").sum()
ch1_running_minutes = (
    is_running(minute_data["CH-1"])
    .resample("1h")
    .sum()
    .reindex(steady_count.index)
    .fillna(0)
)
ch1_off = ch1_running_minutes == 0
ch2_steady_hourly = (
    ch2_minute_data[ch2_minute_data["STEADY"]]
    .resample("1h")
    .mean()
    .reindex(steady_count.index)
)
alone = ch2_steady_hourly[(steady_count >= 45) & ch1_off]
T_SET = chiller.full_load_supply(raw_minutes["CH-2"])  # 現場平常的出水設定
drift = alone["T_CHWS"] > T_SET + 0.5  # 出水拉不下來

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=("CH-2 出水溫度 (°C)", "CH-2 回水溫度 (°C)", "CH-2 耗電 (kW)"),
)
hour_groups = [
    (~drift, "出水正常", "#86b6ef"),
    (drift, f"出水拉不下來（> {T_SET + 0.5:.1f}°C）", "#a33f17"),
]
for hour_mask, group_name, color in hour_groups:
    group_hours = alone[hour_mask]
    for row, column in [(1, "T_CHWS"), (2, "T_CHWR"), (3, "kW")]:
        fig.add_trace(
            go.Scatter(
                x=group_hours["Q_COOL"],
                y=group_hours[column],
                mode="markers",
                name=group_name,
                legendgroup=group_name,
                showlegend=row == 1,
                marker=dict(size=7, color=color, opacity=0.8),
                text=group_hours.index.strftime("%m/%d %H:00"),
                hovertemplate="%{text}｜負荷 %{x:.0f} RT｜%{y:.1f}<extra></extra>",
            ),
            row=row,
            col=1,
        )
fig.add_vline(
    x=rules.PARALLEL_ABOVE,
    line=dict(color="#8a8984", width=1.5, dash="dash"),
)
fig.update_xaxes(title_text="CH-2 負荷 (RT)", row=3, col=1)
show(
    fig,
    "只開 CH-2 時負荷與出水、回水、耗電",
    f"只畫只開 CH-2、穩定運轉的小時；虛線為 {rules.PARALLEL_ABOVE} RT",
    legend="bottom",
    height=850,
)

drift_hours = alone[drift]
print(
    f"只開 CH-2 穩定運轉 {len(alone)} 小時；出水拉不下來 {len(drift_hours)} 小時，"
    f"CH-2 負荷 {drift_hours['Q_COOL'].min():.0f}~{drift_hours['Q_COOL'].max():.0f} RT"
)
print(
    f"  出水 {drift_hours['T_CHWS'].min():.1f}~{drift_hours['T_CHWS'].max():.1f}°C；"
    f"回水 {drift_hours['T_CHWR'].min():.1f}~{drift_hours['T_CHWR'].max():.1f}°C"
    f"（其他小時中位數 {alone.loc[~drift, 'T_CHWR'].median():.1f}°C）；"
    f"耗電 {drift_hours['kW'].min():.0f}~{drift_hours['kW'].max():.0f} kW"
    f"（只開 CH-2 最大 {alone['kW'].max():.0f} kW）"
)
drift_within_limit = drift & (alone["Q_COOL"] <= rules.PARALLEL_ABOVE)
print(
    f"  負荷 ≤ {rules.PARALLEL_ABOVE} RT 時出水拉不下來 {drift_within_limit.sum()} 小時"
)

只開 CH-2 穩定運轉 83 小時；出水拉不下來 10 小時，CH-2 負荷 382~396 RT
  出水 9.2~10.5°C；回水 14.0~15.1°C（其他小時中位數 12.5°C）；耗電 233~252 kW（只開 CH-2 最大 252 kW）
  負荷 ≤ 380 RT 時出水拉不下來 0 小時


### 3.2 理論最佳：各負荷最省電開法與耗電

- **理論最佳**：各負荷下取最省電的開法，並聯時用最佳負荷分配（現場做不到，作為對照）
- **實測點**：每分鐘資料（排除開關機過渡）；單機用該機負荷，兩台同時運轉用合計
- **推估耗電**：各負荷下最省電開法的各機耗電（面積堆疊，虛線為總耗電）
- **右軸**：該開法下各機負荷

In [6]:
fig = go.Figure()
# 面積：建議運轉下各機推估耗電（堆疊即總耗電）
for name, color in [("CH-1", "#b9d4f4"), ("CH-2", "#f9cdb7")]:
    estimated_kw = chiller.chiller_kw(curves[name], table[f"{name} RT"])
    part_kw = pd.Series(estimated_kw, index=table.index).fillna(0)
    fig.add_trace(
        go.Scatter(
            x=table.index,
            y=part_kw,
            mode="lines",
            stackgroup="onoff",
            name=name,
            legend="legend2",
            legendrank=2,
            line=dict(width=0.5, color=color),
            fillcolor=color,
            customdata=table[f"{name} RT"],
            hovertemplate=(
                f"{name}：負荷 %{{customdata:.0f}} RT"
                f"，耗電 %{{y:.0f}} kW<extra></extra>"
            ),
        )
    )

# 實測點
for name, (load_rt, power_kw, color) in points.items():
    fig.add_trace(
        go.Scattergl(
            x=load_rt,
            y=power_kw,
            mode="markers",
            name=name,
            legend="legend",
            marker=dict(size=4, color=color, opacity=0.3),
            text=load_rt.index.strftime("%m/%d %H:%M"),
            hovertemplate=(
                "%{text}<br>負荷：%{x:.0f} RT<br>耗電：%{y:.0f} kW<extra></extra>"
            ),
        )
    )

# 建議運轉總耗電（排在該組圖例最上面）
fig.add_trace(
    go.Scatter(
        x=table.index,
        y=table["預估耗電 kW"],
        mode="lines",
        name="建議運轉總耗電",
        legend="legend2",
        legendrank=1,
        line=dict(color="#a33f17", width=2, dash="dash"),
        customdata=table["建議方案"],
        hovertemplate=(
            "總負荷量 %{x:.0f} RT<br>建議運轉：%{customdata}"
            "<br>總耗電 %{y:.0f} kW<extra></extra>"
        ),
    )
)

# 右軸：建議運轉下各機負荷（未開機不畫）
for name, color in [("CH-1", "#1c5cab"), ("CH-2", "#c84b1a")]:
    fig.add_trace(
        go.Scatter(
            x=table.index,
            y=table[f"{name} RT"].where(table[f"{name} RT"] > 0),
            mode="lines",
            yaxis="y2",
            name=name,
            legend="legend3",
            line=dict(color=color, width=2, dash="dot"),
            customdata=np.stack(
                [table[f"{name} RT"], table[f"{name} 負載率 %"]],
                axis=-1,
            ),
            hovertemplate=(
                f"{name}：負荷 %{{customdata[0]:.0f}} RT"
                f"（負載率 %{{customdata[1]:.0f}}%）<extra></extra>"
            ),
        )
    )

fig.add_vline(
    x=rules.PARALLEL_ABOVE,
    line=dict(color="#8a8984", width=1.5, dash="dash"),
)
fig.add_annotation(
    x=rules.PARALLEL_ABOVE,
    y=1,
    yref="paper",
    text=f"超過 {rules.PARALLEL_ABOVE} RT 並聯",
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    font=dict(size=12, color="#52514e"),
)
fig.update_xaxes(title_text="總負荷量 (RT)")
fig.update_yaxes(title_text="耗電 (kW)")
fig.update_layout(
    yaxis2=dict(
        title="建議運轉負荷 (RT)",
        overlaying="y",
        side="right",
        showgrid=False,
        range=[0, 800],
        tickmode="linear",
        tick0=0,
        dtick=100,
    ),
    margin=dict(r=90),
)
show(
    fig,
    "台數控制：負荷與耗電",
    "面積與虛線為性能曲線推估",
    legend_groups=["實測（每分鐘）", "推估耗電（性能曲線）", "建議運轉負荷（右軸）"],
    height=750,
)

for name, (load_rt, power_kw, _) in points.items():
    print(
        f"{name}：{len(load_rt)} 分鐘，"
        f"負荷 {load_rt.min():.0f}~{load_rt.max():.0f} RT，"
        f"平均 kW/RT {power_kw.sum() / load_rt.sum():.3f}"
    )

# 幾個代表負荷的建議運轉
example_loads = [200, 380, 381, 400, 410, 450, 500, 600]
example_columns = [
    "建議方案",
    "CH-1 RT",
    "CH-1 負載率 %",
    "CH-2 RT",
    "CH-2 負載率 %",
    "預估耗電 kW",
]
table.loc[example_loads, example_columns].round(1)

CH-2：7412 分鐘，負荷 146~414 RT，平均 kW/RT 0.572
CH-1：1365 分鐘，負荷 94~198 RT，平均 kW/RT 1.241
CH-1＋CH-2：1365 分鐘，負荷 375~549 RT，平均 kW/RT 0.786


,建議方案,CH-1 RT,CH-1 負載率 %,CH-2 RT,CH-2 負載率 %,預估耗電 kW
全館負荷 RT,,,,,,
200.0,CH-2,0.0,0.0,200.0,50.0,96.5
380.0,CH-2,0.0,0.0,380.0,95.0,236.8
381.0,CH-1 + CH-2,179.0,47.7,202.0,50.5,286.3
400.0,CH-1 + CH-2,198.0,52.8,202.0,50.5,292.0
410.0,CH-1 + CH-2,208.0,55.5,202.0,50.5,295.1
450.0,CH-1 + CH-2,248.0,66.1,202.0,50.5,307.3
500.0,CH-1 + CH-2,298.0,79.5,202.0,50.5,322.6
600.0,CH-1 + CH-2,375.0,100.0,225.0,56.2,355.4


### 3.3 現行規則：各負荷運轉模式與耗電

- **現行規則**：≤ 380 RT 單機（CH-2）、> 380 RT 兩台並聯；並聯時兩台同一出水設定，負荷依兩台冰水流量比例分配（面積堆疊，實線為總耗電）
- **理論最佳總耗電**（灰色虛線）：同上一張圖，作為對照
- 實測點、右軸與上一張圖相同

In [7]:
# 並聯時 CH-1 流量占比：兩台同時穩定運轉的分鐘之中位數
ch1_flow = ch1_minutes.loc[both_steady, "FLOW"]
ch2_flow = ch2_minutes.loc[both_steady, "FLOW"]
share = (ch1_flow / (ch1_flow + ch2_flow)).median()

# 現行規則下各機負荷與耗電：單機全由 CH-2 承擔；並聯依流量比例分配
loads = table.index[table.index >= rules.minimum_load_rt(curves["CH-2"])]
plan = rules.staging(pd.Series(loads, index=loads))
rule_table = pd.DataFrame(
    {
        "CH-1 RT": np.where(plan == rules.BOTH, loads * share, 0.0),
        "CH-2 RT": np.where(plan == rules.BOTH, loads * (1 - share), loads),
    },
    index=loads,
)
for name in ["CH-1", "CH-2"]:
    rule_table[f"{name} kW"] = np.where(
        rule_table[f"{name} RT"] > 0,
        chiller.chiller_kw(curves[name], rule_table[f"{name} RT"]),
        0.0,
    )
# 任一台超出運轉範圍為 NaN
rule_table["總耗電 kW"] = rule_table["CH-1 kW"] + rule_table["CH-2 kW"]
rule_table = rule_table[rule_table["總耗電 kW"].notna()]

fig = go.Figure()
# 面積：現行規則下各機耗電（堆疊即總耗電）
for name, color in [("CH-1", "#b9d4f4"), ("CH-2", "#f9cdb7")]:
    fig.add_trace(
        go.Scatter(
            x=rule_table.index,
            y=rule_table[f"{name} kW"],
            mode="lines",
            stackgroup="rule",
            name=name,
            legend="legend2",
            legendrank=3,
            line=dict(width=0.5, color=color),
            fillcolor=color,
            customdata=rule_table[f"{name} RT"],
            hovertemplate=(
                f"{name}：負荷 %{{customdata:.0f}} RT"
                f"，耗電 %{{y:.0f}} kW<extra></extra>"
            ),
        )
    )

# 實測點
for name, (load_rt, power_kw, color) in points.items():
    fig.add_trace(
        go.Scattergl(
            x=load_rt,
            y=power_kw,
            mode="markers",
            name=name,
            legend="legend",
            marker=dict(size=4, color=color, opacity=0.3),
            text=load_rt.index.strftime("%m/%d %H:%M"),
            hovertemplate=(
                "%{text}<br>負荷：%{x:.0f} RT<br>耗電：%{y:.0f} kW<extra></extra>"
            ),
        )
    )

# 現行規則總耗電與理論最佳對照
fig.add_trace(
    go.Scatter(
        x=rule_table.index,
        y=rule_table["總耗電 kW"],
        mode="lines",
        name="現行規則總耗電",
        legend="legend2",
        legendrank=1,
        line=dict(color="#a33f17", width=2),
        hovertemplate="總負荷量 %{x:.0f} RT<br>現行規則 %{y:.0f} kW<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=table.index,
        y=table["預估耗電 kW"],
        mode="lines",
        name="理論最佳總耗電",
        legend="legend2",
        legendrank=2,
        line=dict(color="#8a8984", width=2, dash="dash"),
        hovertemplate="理論最佳 %{y:.0f} kW<extra></extra>",
    )
)

# 右軸：現行規則下各機負荷（未開機不畫）
for name, color in [("CH-1", "#1c5cab"), ("CH-2", "#c84b1a")]:
    fig.add_trace(
        go.Scatter(
            x=rule_table.index,
            y=rule_table[f"{name} RT"].where(rule_table[f"{name} RT"] > 0),
            mode="lines",
            yaxis="y2",
            name=name,
            legend="legend3",
            line=dict(color=color, width=2, dash="dot"),
            customdata=rule_table[f"{name} RT"] / curves[name]["rated"] * 100,
            hovertemplate=(
                f"{name}：負荷 %{{y:.0f}} RT"
                f"（負載率 %{{customdata:.0f}}%）<extra></extra>"
            ),
        )
    )

fig.add_vline(
    x=rules.PARALLEL_ABOVE,
    line=dict(color="#8a8984", width=1.5, dash="dash"),
)
fig.add_annotation(
    x=rules.PARALLEL_ABOVE,
    y=1,
    yref="paper",
    text=f"超過 {rules.PARALLEL_ABOVE} RT 並聯",
    showarrow=False,
    xanchor="right",
    yanchor="bottom",
    font=dict(size=12, color="#52514e"),
)
fig.update_xaxes(title_text="總負荷量 (RT)")
fig.update_yaxes(title_text="耗電 (kW)")
fig.update_layout(
    yaxis2=dict(
        title="現行規則負荷 (RT)",
        overlaying="y",
        side="right",
        showgrid=False,
        range=[0, 800],
        tickmode="linear",
        tick0=0,
        dtick=100,
    ),
    margin=dict(r=90),
)
show(
    fig,
    "台數控制：現行規則的負荷與耗電",
    f"並聯時負荷依流量比例分配（CH-1 {share:.0%}、CH-2 {1 - share:.0%}）；"
    f"面積與實線為性能曲線推估",
    legend_groups=["實測（每分鐘）", "推估耗電（性能曲線）", "現行規則負荷（右軸）"],
    height=750,
)

print(
    f"並聯時流量占比（中位數）：CH-1 {share:.0%}、CH-2 {1 - share:.0%}；"
    f"現行規則可推估至 {rule_table.index.max():.0f} RT"
)

# 幾個代表負荷：現行規則與理論最佳的耗電
comparison = pd.DataFrame(
    {
        "現行規則 kW": rule_table["總耗電 kW"],
        "理論最佳 kW": table["預估耗電 kW"],
    }
).loc[[200, 380, 381, 400, 450, 500, 600]]
comparison["差 kW"] = comparison["現行規則 kW"] - comparison["理論最佳 kW"]
comparison.round(1)

並聯時流量占比（中位數）：CH-1 46%、CH-2 54%；現行規則可推估至 737 RT


,現行規則 kW,理論最佳 kW,差 kW
全館負荷 RT,,,
200.0,96.5,96.5,0.0
380.0,236.8,236.8,0.0
381.0,286.3,286.3,0.1
400.0,293.0,292.0,1.0
450.0,315.4,307.3,8.1
500.0,344.3,322.6,21.7
600.0,412.8,355.4,57.3


## 4. 台數控制規則

以負荷模型之逐時預測冷負荷（全館）決定運轉台數；參數設於 `.env`，規則程式為 `model/onoff.py` 的 `staging`、`hold_staging`。

### 4.1 運轉模式判定

| 預測冷負荷 | 運轉模式 | 依據 |
|---|---|---|
| < 60 RT | 停機 | 低於 CH-2 最低負載限制（額定 400 RT × 最低負載率 15% = 60 RT）；主機在最低負載以下無法穩定運轉，故不開機 |
| 60 ~ 380 RT | 單機（CH-2） | CH-2 部分負載效率優於 CH-1（第 2 節） |
| > 380 RT（`CONTROL_PARALLEL_ABOVE`） | 兩台並聯，同一出水設定 | 單機負荷超過 380 RT 時已達額定輸入功率，無法維持供水溫度（第 3 節） |

### 4.2 最短運轉時段（防止頻繁加卸載）

預測負荷在門檻附近波動時，若逐時直接依門檻判定，會造成主機頻繁加卸載，增加啟停次數與壓縮機磨耗，並造成供水溫度擾動。因此同一日內，若某一運轉模式持續時數 ≤ 3 小時（`CONTROL_MIN_STAGE_HOURS`），且前後時段為相同的另一模式，該時段即併入前後模式：
- 單機時段中短暫並聯 → 維持單機
- 並聯時段中短暫單機 → 維持並聯
- 多段同時符合時，由持續時數最短者先處理；時數相同時優先改為並聯（供冷優先）
- 前後模式不同的單純轉換（如單機轉並聯後持續運轉）不受影響

取捨：單機時段中短暫並聯併入單機時，該時段預測負荷雖超過 380 RT，仍由 CH-2 單機承擔，CH-2 可能短時間無法維持供水溫度設定。這是為了減少加卸載次數而接受的風險。

### 4.3 並聯負荷分配

並聯時兩台僅能設定同一出水溫度，負荷依兩台冰水流量比例分配（CH-1 約 46%、CH-2 約 54%），無法依效率做最佳分配。

In [8]:
# 規則參數（.env）
ch2_curve = curves["CH-2"]
print(
    f"停機門檻：CH-2 最低負載 {rules.minimum_load_rt(ch2_curve):.0f} RT"
    f"（額定 {ch2_curve['rated']} RT × 最低負載率 {ch2_curve['min_lr']}%）"
)
print(
    f"並聯門檻：> {rules.PARALLEL_ABOVE} RT；"
    f"最短運轉時段：≤ {rules.MIN_STAGE_HOURS} 小時"
)
print(f"並聯時流量占比（中位數）：CH-1 {share:.0%}、CH-2 {1 - share:.0%}")

停機門檻：CH-2 最低負載 60 RT（額定 400 RT × 最低負載率 15%）
並聯門檻：> 380 RT；最短運轉時段：≤ 3 小時
並聯時流量占比（中位數）：CH-1 46%、CH-2 54%


## 5. 結論與限制

**性能曲線**
- CH-1：kW/RT = 0.305 + 35.9 ÷ 負載率（R² 0.98）；滿載外推約 0.66 kW/RT，與 Trane RTHD 型錄（0.55~0.69）相當
- CH-2：三次式（R² 0.69）；負載率約 55% 效率最佳（COP 約 7.4），滿載約 0.60 kW/RT，與型錄 KLTW-400D-3（0.576 kW/RT）相近

**台數**
- **預測冷負荷 < 60 RT（CH-2 最低負載限制）停機、60~380 RT 單機（CH-2）、> 380 RT 兩台並聯**；同一日內 ≤ 3 小時且前後模式相同的短時段併入前後模式

**限制**
- CH-1 型號未確認（推測 Trane RTHD／RTHC，約 370~377 RT）
- 負載率低於資料範圍（CH-1 < 24%、CH-2 < 46%）及 CH-1 高於 83% 為外推